# **LABORATORIO N 11 - Gestion de Metadatos**

**Curso:** Fundamentos de Gestion de Datos  
**Docente:** Pilar Rocio Sayan Mejia  
**Semana:** 11  
**Tema:** Construccion del Catalogo de Metadatos del Proyecto

---

## **Actividad 1: Revision de Conceptos**

Complete la siguiente tabla con sus propias palabras:

| Concepto | Definicion |
|----------|------------|
| 1. Que son los metadatos? | |
| 2. Tipos de metadatos | |
| 3. Que es un catalogo de datos? | |
| 4. Que es un diccionario de datos? | |
| 5. Que es la trazabilidad del dato? | |
| 6. Que es el linaje de datos? | |
| 7. Que es el estandar Dublin Core? | |
| 8. Que es el estandar ISO 11179? | |
| 9. Metadatos descriptivos vs estructurales | |
| 10. Que es la gobernanza de datos? | |
| 11. Importancia de documentar reglas de negocio | |
| 12. Beneficios del catalogo de metadatos | |

---

## **Actividad 2: Desarrollo Practico - Construccion del Catalogo de Metadatos**

En esta actividad construiremos un catalogo de metadatos completo.

### **Paso 1: Configuracion del entorno**

In [ ]:
# Importar librerias necesarias
import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Librerias importadas correctamente")

### **Paso 2: Crear base de datos de ejemplo**

In [ ]:
# Crear base de datos de ejemplo
conn = sqlite3.connect(':memory:')

# Crear tabla de clientes
conn.execute('''
    CREATE TABLE clientes (
        id INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email TEXT UNIQUE,
        telefono TEXT,
        fecha_registro TEXT,
        activo INTEGER DEFAULT 1
    )
''')

# Crear tabla de productos
conn.execute('''
    CREATE TABLE productos (
        id INTEGER PRIMARY KEY,
        codigo TEXT UNIQUE NOT NULL,
        nombre TEXT NOT NULL,
        descripcion TEXT,
        precio REAL NOT NULL,
        stock INTEGER DEFAULT 0,
        categoria_id INTEGER
    )
''')

# Crear tabla de ventas
conn.execute('''
    CREATE TABLE ventas (
        id INTEGER PRIMARY KEY,
        cliente_id INTEGER,
        producto_id INTEGER,
        cantidad INTEGER NOT NULL,
        precio_unitario REAL NOT NULL,
        total REAL,
        fecha TEXT,
        FOREIGN KEY (cliente_id) REFERENCES clientes(id),
        FOREIGN KEY (producto_id) REFERENCES productos(id)
    )
''')

# Insertar datos de ejemplo
conn.execute("INSERT INTO clientes VALUES (1, 'Juan Perez', 'juan@email.com', '999888777', '2026-01-15', 1)")
conn.execute("INSERT INTO productos VALUES (1, 'PROD001', 'Laptop', 'Laptop HP 15 pulgadas', 2500.00, 10, 1)")
conn.execute("INSERT INTO ventas VALUES (1, 1, 1, 2, 2500.00, 5000.00, '2026-02-20')")
conn.commit()

print("Base de datos de ejemplo creada con 3 tablas")

### **Paso 3: Obtener estructura de la base de datos**

In [ ]:
# Obtener lista de tablas
print("="*60)
print("ESTRUCTURA DE LA BASE DE DATOS")
print("="*60)

query_tablas = "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
tablas = pd.read_sql_query(query_tablas, conn)

print("\nTablas en la base de datos:")
for t in tablas['name']:
    print(f"  - {t}")

print(f"\nTotal de tablas: {len(tablas)}")

In [ ]:
# Funcion para obtener metadatos de una tabla
def obtener_metadatos_tabla(conn, tabla):
    """Obtiene informacion detallada de las columnas de una tabla"""
    query = f"PRAGMA table_info({tabla})"
    columnas = pd.read_sql_query(query, conn)
    return columnas

# Mostrar estructura de cada tabla
for tabla in tablas['name']:
    print(f"\n{'='*50}")
    print(f"TABLA: {tabla}")
    print('='*50)
    metadatos = obtener_metadatos_tabla(conn, tabla)
    display(metadatos)

**Pregunta 1:** Cuantas tablas tiene tu base de datos? Lista sus nombres.

**Respuesta:** _Escribe tu respuesta aqui_

**Pregunta 2:** Cuantas columnas tiene cada tabla? Cual es el tipo de dato de cada una?

**Respuesta:** _Escribe tu respuesta aqui_

### **Paso 4: Crear plantilla de catalogo de metadatos**

In [ ]:
# Crear catalogo de metadatos
print("="*60)
print("CREANDO CATALOGO DE METADATOS")
print("="*60)

catalogo_columnas = []

for tabla in tablas['name']:
    columnas = obtener_metadatos_tabla(conn, tabla)
    
    for _, col in columnas.iterrows():
        catalogo_columnas.append({
            'tabla': tabla,
            'columna': col['name'],
            'tipo_dato': col['type'],
            'permite_nulo': 'Si' if col['notnull'] == 0 else 'No',
            'es_pk': 'Si' if col['pk'] == 1 else 'No',
            'valor_por_defecto': col['dflt_value'],
            'descripcion': '',
            'fuente': '',
            'regla_negocio': ''
        })

df_catalogo = pd.DataFrame(catalogo_columnas)
print(f"\nCatalogo creado con {len(df_catalogo)} columnas documentadas")
print("\nPrimeras 10 filas:")
df_catalogo.head(10)

**Pregunta 3:** Muestra las primeras 10 filas del catalogo generado. Que informacion automatica obtuviste?

**Respuesta:** _Escribe tu respuesta aqui_

### **Paso 5: Documentar descripciones y reglas de negocio**

In [ ]:
# Definir descripciones para cada columna
descripciones = {
    # Clientes
    ('clientes', 'id'): 'Identificador unico del cliente',
    ('clientes', 'nombre'): 'Nombre completo del cliente',
    ('clientes', 'email'): 'Correo electronico del cliente',
    ('clientes', 'telefono'): 'Numero de telefono del cliente',
    ('clientes', 'fecha_registro'): 'Fecha en que el cliente se registro',
    ('clientes', 'activo'): 'Indica si el cliente esta activo (1) o inactivo (0)',
    
    # Productos
    ('productos', 'id'): 'Identificador unico del producto',
    ('productos', 'codigo'): 'Codigo interno del producto',
    ('productos', 'nombre'): 'Nombre comercial del producto',
    ('productos', 'descripcion'): 'Descripcion detallada del producto',
    ('productos', 'precio'): 'Precio de venta del producto',
    ('productos', 'stock'): 'Cantidad disponible en inventario',
    ('productos', 'categoria_id'): 'Identificador de la categoria del producto',
    
    # Ventas
    ('ventas', 'id'): 'Identificador unico de la venta',
    ('ventas', 'cliente_id'): 'Referencia al cliente que realizo la compra',
    ('ventas', 'producto_id'): 'Referencia al producto vendido',
    ('ventas', 'cantidad'): 'Cantidad de unidades vendidas',
    ('ventas', 'precio_unitario'): 'Precio por unidad al momento de la venta',
    ('ventas', 'total'): 'Monto total de la venta (cantidad x precio_unitario)',
    ('ventas', 'fecha'): 'Fecha en que se realizo la venta'
}

# Definir fuentes de datos
fuentes = {
    ('clientes', 'id'): 'Sistema interno - Autoincremental',
    ('clientes', 'nombre'): 'Registro de cliente',
    ('clientes', 'email'): 'Registro de cliente',
    ('productos', 'precio'): 'Sistema de precios - Area comercial',
    ('ventas', 'total'): 'Calculado: cantidad * precio_unitario',
}

# Definir reglas de negocio
reglas = {
    ('clientes', 'email'): 'Debe tener formato valido de email',
    ('clientes', 'activo'): 'Valor por defecto: 1 (activo)',
    ('productos', 'precio'): 'Debe ser mayor a 0',
    ('productos', 'stock'): 'No puede ser negativo',
    ('ventas', 'cantidad'): 'Debe ser mayor a 0',
    ('ventas', 'total'): 'Calculado automaticamente'
}

In [ ]:
# Actualizar catalogo con la documentacion
for idx, row in df_catalogo.iterrows():
    key = (row['tabla'], row['columna'])
    
    if key in descripciones:
        df_catalogo.at[idx, 'descripcion'] = descripciones[key]
    
    if key in fuentes:
        df_catalogo.at[idx, 'fuente'] = fuentes[key]
    
    if key in reglas:
        df_catalogo.at[idx, 'regla_negocio'] = reglas[key]

print("Catalogo actualizado con descripciones y reglas")
print("\nCatalogo completo:")
df_catalogo

**Pregunta 4:** Documenta al menos 5 columnas con su descripcion, fuente y regla de negocio.

**Respuesta:** _Escribe tu respuesta aqui_

### **Paso 6: Guardar catalogo de metadatos**

In [ ]:
# Guardar catalogo como tabla en SQLite
df_catalogo.to_sql('catalogo_metadatos', conn, if_exists='replace', index=False)

print("Catalogo guardado en SQLite como tabla 'catalogo_metadatos'")

# Verificar guardado
df_verificacion = pd.read_sql_query('SELECT * FROM catalogo_metadatos', conn)
print(f"\nRegistros guardados: {len(df_verificacion)}")

In [ ]:
# Resumen del catalogo
print("="*60)
print("RESUMEN DEL CATALOGO DE METADATOS")
print("="*60)

print(f"\nTotal de tablas documentadas: {df_catalogo['tabla'].nunique()}")
print(f"Total de columnas documentadas: {len(df_catalogo)}")
print(f"\nColumnas por tabla:")
print(df_catalogo.groupby('tabla').size())

print(f"\nColumnas con descripcion: {(df_catalogo['descripcion'] != '').sum()}")
print(f"Columnas con fuente: {(df_catalogo['fuente'] != '').sum()}")
print(f"Columnas con regla de negocio: {(df_catalogo['regla_negocio'] != '').sum()}")

---

## **Actividad 3: Caso de Estudio - Catalogo del Proyecto Grupal**

Construye el catalogo de metadatos de tu proyecto grupal.

In [ ]:
# ESPACIO PARA TU CATALOGO DE METADATOS DEL PROYECTO

# # Conectar a tu base de datos
# conn_proyecto = sqlite3.connect('tu_proyecto.db')

# # Obtener tablas
# tablas_proyecto = pd.read_sql_query(
#     "SELECT name FROM sqlite_master WHERE type='table'", 
#     conn_proyecto
# )

# # Crear catalogo
# catalogo_proyecto = []
# for tabla in tablas_proyecto['name']:
#     columnas = obtener_metadatos_tabla(conn_proyecto, tabla)
#     # Documentar cada columna...
# 
# # Guardar catalogo
# df_catalogo_proyecto.to_sql('catalogo_metadatos', conn_proyecto, if_exists='replace', index=False)

**Pregunta A:** Como mejoraria tener un catalogo de metadatos el trabajo de tu equipo? Menciona al menos 2 situaciones donde lo hubieran necesitado.

**Respuesta:** _Escribe tu respuesta aqui_

**Pregunta B:** Muestra el catalogo completo con todas las tablas y columnas documentadas.

**Respuesta:** _Escribe tu respuesta aqui_

**Pregunta C:** Que estandar de metadatos (Dublin Core, ISO 11179) podria aplicar a tu proyecto? Por que?

**Respuesta:** _Escribe tu respuesta aqui_

---

## **Conclusiones**

Escribe tus conclusiones tecnicas y de aprendizaje:

1. _Escribe tu primera conclusion aqui_

2. _Escribe tu segunda conclusion aqui_

3. _Escribe tu tercera conclusion aqui_

In [ ]:
# Cerrar conexion
conn.close()
print("Conexion cerrada correctamente")